# 09 - Ablation of the DFM components M0-M3 (workflow step 11b)

* $\mathcal M_0$ -- full DFM.
* $\mathcal M_1$ -- without the health/degradation limitation (nameplate energy in the envelope
  and EMS, no $\mathcal F^{deg}$; the plant still ages with its true SOH).
* $\mathcal M_2$ -- without the uncertainty reserve ($P_{res}=0$).
* $\mathcal M_3$ -- without the AC/DC network envelope ($\mathcal F^{net}$ and the network-required
  curtailment model removed).

Evaluated nominally and on the same 100 Monte Carlo realizations as notebook 08
(M0 trials are taken from there). Relative change $\Delta z_\% = 100(z_{\mathcal M_i}-z_{\mathcal M_0})/z_{\mathcal M_0}$.

Outputs: `output/results/results_ablation_trials.csv`, `output/results/results_ablation_summary.csv`.

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
import pandas as pd

from dfm_acdc.config import RESULTS_DIR, MCParams
from dfm_acdc.mc import run_jobs

PRE_PATH = os.path.join(RESULTS_DIR, "precompute_network_forecast.csv")
mc = MCParams()
if os.environ.get("DFM_SMOKE"):
    mc.n_trials = 2          # quick end-to-end smoke test only

In [ ]:
t0 = time.perf_counter()
jobs = [{"strategy": s, "trial": None, "tag": "abl_nominal"} for s in ("DFM", "M1", "M2", "M3")]
jobs += [{"strategy": s, "trial": t, "tag": "abl_mc"} for t in range(mc.n_trials) for s in ("M1", "M2", "M3")]
out = run_jobs(jobs, PRE_PATH, mc.seed, mc.n_trials)
abl = pd.DataFrame([o["summary"] for o in out])
main = pd.read_csv(os.path.join(RESULTS_DIR, "results_mc_trials.csv"))
m0 = main[main.strategy == "DFM"].copy(); m0["tag"] = "abl_mc"
abl = pd.concat([abl, m0], ignore_index=True)
abl["config"] = abl.strategy.replace({"DFM": "M0"})
abl.to_csv(os.path.join(RESULTS_DIR, "results_ablation_trials.csv"), index=False)
print(f"{len(out)} runs in {time.perf_counter()-t0:.0f} s")

In [ ]:
M = ["E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "N_EFC", "E_dev_kWh",
     "n_viol_intervals", "max_exceedance", "soc_final", "t_solve_mean_ms", "t_solve_p95_ms",
     "share_lim_dis_deg", "share_lim_ch_deg", "mean_P_safe_dis_kW", "mean_P_safe_ch_kW"]
rows = []
for cfg in ("M0", "M1", "M2", "M3"):
    nom = abl[(abl.config == cfg) & (abl.tag == "abl_nominal")].iloc[0]
    d = abl[(abl.config == cfg) & (abl.tag == "abl_mc")]
    r = {"config": cfg, "n_trials": len(d)}
    for m in M:
        r[f"nom_{m}"] = nom[m]
        r[f"mc_mean_{m}"] = d[m].mean()
        r[f"mc_P95_{m}"] = np.percentile(d[m], 95)
    r["mc_P_viol_trials"] = float((d.n_viol_intervals > 0).mean())
    r["mc_t_solve_p95_ms_pooled"] = float(np.percentile(d.t_solve_p95_ms, 95))
    rows.append(r)
summ = pd.DataFrame(rows).set_index("config")
for m in ("E_grid_corr_kWh", "cost_corr_EUR", "E_curt_kWh", "dSOH_rainflow", "E_dev_kWh"):
    summ[f"rel_mc_{m}_pct"] = 100 * (summ[f"mc_mean_{m}"] - summ.loc["M0", f"mc_mean_{m}"]) / summ.loc["M0", f"mc_mean_{m}"]
summ.to_csv(os.path.join(RESULTS_DIR, "results_ablation_summary.csv"))
show = ["nom_cost_corr_EUR", "nom_E_curt_kWh", "nom_dSOH_rainflow", "nom_n_viol_intervals",
        "mc_mean_cost_corr_EUR", "mc_mean_E_curt_kWh", "mc_mean_dSOH_rainflow", "mc_mean_E_dev_kWh",
        "mc_P_viol_trials", "mc_P95_max_exceedance", "mc_mean_soc_final", "mc_mean_t_solve_mean_ms",
        "mc_t_solve_p95_ms_pooled", "rel_mc_cost_corr_EUR_pct", "rel_mc_E_curt_kWh_pct",
        "rel_mc_dSOH_rainflow_pct", "rel_mc_E_dev_kWh_pct", "mc_mean_share_lim_dis_deg"]
print(summ[show].T.to_string())